# Holiday Package Prediction using. Multiple Models and implementating Random Forest

## Holiday Package Prediciton

### 1) Problem statement.
"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base.
One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering * Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information.
The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being.
However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.
### 2) Data Collection.
The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction
The data consists of 20 column and 4888 rows.


In [62]:
# import all the necessary librariess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [63]:
# Load the dataset
dataset=pd.read_csv('dataset/Travel.csv')

In [64]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4717,204717,0,35.0,Company Invited,1,10.0,Salaried,Male,3,5.0,Basic,3.0,Married,5.0,0,1,0,1.0,Executive,21657.0
3160,203160,0,42.0,Company Invited,1,16.0,Small Business,Male,4,4.0,King,3.0,Married,NaN,0,4,1,3.0,VP,38097.0
4879,204879,1,26.0,Self Enquiry,2,27.0,Small Business,Female,4,4.0,Basic,4.0,Married,2.0,1,3,0,2.0,Executive,22347.0
2482,202482,0,37.0,Self Enquiry,1,12.0,Salaried,Female,3,5.0,Basic,5.0,Divorced,2.0,1,2,1,1.0,Executive,98678.0
1065,201065,0,NaN,Self Enquiry,1,10.0,Salaried,Male,1,3.0,Deluxe,3.0,Divorced,1.0,1,4,0,0.0,Manager,NaN


In [65]:
# Data Cleaning
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [66]:
dataset.isna().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [67]:
dataset[dataset.isna().any(axis=1)]

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
11,200011,0,NaN,Self Enquiry,1,21.0,Salaried,Female,2,4.0,Deluxe,3.0,Single,1.0,1,3,0,0.0,Manager,NaN
19,200019,0,NaN,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Basic,3.0,Single,6.0,1,4,0,1.0,Executive,NaN
20,200020,0,NaN,Company Invited,1,17.0,Salaried,Female,3,2.0,Deluxe,3.0,Married,1.0,0,3,1,2.0,Manager,NaN
21,200021,1,NaN,Self Enquiry,3,15.0,Salaried,Male,2,4.0,Deluxe,5.0,Single,1.0,0,2,0,0.0,Manager,18407.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4850,204850,1,46.0,Self Enquiry,3,8.0,Salaried,Male,4,5.0,Deluxe,5.0,Married,NaN,0,4,1,3.0,Manager,36739.0
4851,204851,1,40.0,Self Enquiry,1,9.0,Salaried,Female,4,4.0,Basic,5.0,Married,NaN,1,1,1,1.0,Executive,35801.0
4868,204868,1,43.0,Company Invited,2,15.0,Salaried,Female,4,5.0,Basic,3.0,Married,NaN,0,5,1,2.0,Executive,36539.0
4869,204869,1,56.0,Self Enquiry,3,16.0,Small Business,Female,3,6.0,Basic,4.0,Single,NaN,0,1,1,2.0,Executive,37865.0


In [68]:
dataset.duplicated().sum()

np.int64(0)

In [69]:
# Split to categorical and numerical columns
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [70]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
215,200215,0,41.0,Self Enquiry,3,12.0,Salaried,Fe Male,3,3.0,Standard,3.0,Single,4.0,1,2,0,1.0,Senior Manager,28591.0
2693,202693,0,46.0,Company Invited,1,14.0,Salaried,Male,4,3.0,Deluxe,3.0,Divorced,6.0,0,2,0,3.0,Manager,25112.0
3930,203930,0,33.0,Self Enquiry,1,12.0,Salaried,Male,4,2.0,Basic,4.0,Married,2.0,1,1,1,3.0,Executive,21976.0
1237,201237,1,32.0,Self Enquiry,3,6.0,Salaried,Male,2,5.0,Deluxe,4.0,Unmarried,7.0,0,3,0,1.0,Manager,21735.0
4196,204196,0,30.0,Company Invited,3,9.0,Salaried,Male,3,4.0,Deluxe,3.0,Unmarried,3.0,0,1,0,2.0,Manager,23232.0


In [71]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [72]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [73]:
dataset['Gender']=dataset['Gender'].str.replace('Fe Male','Female')

In [74]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [75]:
dataset['MaritalStatus']=dataset['MaritalStatus'].str.replace('Single','Unmarried')

In [76]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [77]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4784,204784,0,36.0,Self Enquiry,3,24.0,Salaried,Male,4,4.0,Deluxe,5.0,Married,5.0,1,3,0,3.0,Manager,27644.0
1275,201275,0,51.0,Self Enquiry,1,9.0,Large Business,Female,2,3.0,Standard,3.0,Married,3.0,0,4,1,1.0,Senior Manager,28116.0
2603,202603,0,35.0,Company Invited,3,9.0,Small Business,Female,4,4.0,Basic,3.0,Divorced,8.0,0,5,1,3.0,Executive,20909.0
3815,203815,0,57.0,Self Enquiry,3,18.0,Small Business,Female,3,5.0,Deluxe,5.0,Married,6.0,0,5,0,2.0,Manager,24058.0
618,200618,0,NaN,Self Enquiry,1,8.0,Small Business,Male,3,1.0,Basic,5.0,Unmarried,1.0,0,2,1,2.0,Executive,18424.0


In [80]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
dataset['TypeofContact'] = dataset['TypeofContact'].fillna(dataset['TypeofContact'].mode()[0])
dataset['DurationOfPitch'] = dataset['DurationOfPitch'].fillna(dataset['DurationOfPitch'].median())
dataset['NumberOfFollowups'] = dataset['NumberOfFollowups'].fillna(dataset['NumberOfFollowups'].mode()[0])
dataset['PreferredPropertyStar'] = dataset['PreferredPropertyStar'].fillna(dataset['PreferredPropertyStar'].mode()[0])
dataset['NumberOfTrips'] = dataset['NumberOfTrips'].fillna(dataset['NumberOfTrips'].median())
dataset['NumberOfChildrenVisiting'] = dataset['NumberOfChildrenVisiting'].fillna(dataset['NumberOfChildrenVisiting'].mode()[0])
dataset['MonthlyIncome'] = dataset['MonthlyIncome'].fillna(dataset['MonthlyIncome'].median())


In [82]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
558,200558,0,32.0,Company Invited,1,30.0,Salaried,Male,3,3.0,Deluxe,3.0,Divorced,2.0,0,4,1,0.0,Manager,20309.0
703,200703,0,38.0,Company Invited,3,16.0,Large Business,Female,2,3.0,Deluxe,3.0,Unmarried,2.0,1,3,0,1.0,Manager,20666.0
317,200317,1,52.0,Self Enquiry,1,14.0,Small Business,Male,2,4.0,Deluxe,4.0,Divorced,3.0,0,2,1,1.0,Manager,19941.0
966,200966,1,58.0,Self Enquiry,1,13.0,Small Business,Female,2,4.0,Standard,5.0,Divorced,1.0,1,4,1,0.0,Senior Manager,25008.0
1210,201210,0,33.0,Self Enquiry,1,27.0,Small Business,Male,3,1.0,Basic,4.0,Married,2.0,0,1,1,1.0,Executive,17028.0


In [83]:
dataset.isna().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64